# 🔑 Solutions: Lesson 4: Feature Engineering
**Try the problems yourself first!** Looking at a solution before struggling is like watching someone else do push-ups.
Every solution below is self-contained and has a short explanation of *why* it works.

In [1]:
import sys
sys.path.append("../utils")            # lets us import course_utils.py and agri_prep.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_utils import *             # load_raw_farm, profile, setup_plots, ...

setup_plots()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
np.random.seed(42)


df = load_lesson_clean()
prices = load_prices().sort_values(["crop", "date"]).reset_index(drop=True)

---
### Problem 4.1 &nbsp; ⭐ &nbsp; Date features and the circular month

Using `load_lesson_clean()`:

1. Create `plant_year`, `plant_month` and `grow_days` (harvest − planting, in days). Report the **mean `grow_days` per crop**.
2. Encode the month with sine and cosine (`sin(2π·month/12)`, `cos(2π·month/12)`). Compute the Euclidean distance between the points for **December vs January** and for **January vs July**. Which is smaller, and why does that matter?

In [2]:
df = load_lesson_clean()
df["plant_year"]  = df["planting_date"].dt.year
df["plant_month"] = df["planting_date"].dt.month
df["grow_days"]   = (df["harvest_date"] - df["planting_date"]).dt.days
print(df.groupby("crop")["grow_days"].mean().round(0).to_string())

point = lambda m: np.array([np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)])
d = lambda a, b: float(np.linalg.norm(point(a) - point(b)))
print(f"\nDec↔Jan distance: {d(12, 1):.2f}   Jan↔Jul distance: {d(1, 7):.2f}   (plain numbers: 11 vs 6)")

crop
Beans       90.0
Maize      160.0
Potato     110.0
Sorghum    120.0
Wheat      130.0

Dec↔Jan distance: 0.52   Jan↔Jul distance: 2.00   (plain numbers: 11 vs 6)


**💬 Why this works:** Plain month numbers say December (12) is *far* from January (1). On the circle they are neighbours (0.52 apart) while January and July are opposite (2.0). Cyclical features let distance-based and linear models understand that time wraps around.

---
### Problem 4.2 &nbsp; ⭐⭐ &nbsp; Which engineered ratio matters most?

Restrict to **Maize** rows. Create three features:
`npk_total = N + P + K`, `n_to_p_ratio = N / (P + 1)` and `moisture_index = rainfall / temperature`.

Compute the absolute correlation of each (and of the raw `rainfall_mm`) with `log(yield_t_ha)`. Which engineered feature, if any, beats the raw rainfall? Why do we add 1 to the denominator in the ratio?

In [3]:
df = load_lesson_clean()
m = df[df["crop"] == "Maize"].copy()
m["npk_total"]      = m["nitrogen_ppm"] + m["phosphorus_ppm"] + m["potassium_ppm"]
m["n_to_p_ratio"]   = m["nitrogen_ppm"] / (m["phosphorus_ppm"] + 1)
m["moisture_index"] = m["rainfall_mm"] / m["avg_temp_c"]

target = np.log(m["yield_t_ha"])
cols = ["rainfall_mm", "npk_total", "n_to_p_ratio", "moisture_index"]
print(m[cols].corrwith(target).abs().sort_values(ascending=False).round(3).to_string())

moisture_index    0.252
rainfall_mm       0.102
npk_total         0.045
n_to_p_ratio      0.021


**💬 Why this works:** Feature engineering is a *hypothesis*: 'maybe the balance of nutrients matters'. Correlation gives a first (linear!) check. The +1 protects against division by zero or near-zero denominators, which would create huge or infinite values. Tip: a weak *linear* correlation doesn't prove a feature useless: yield here depends on rainfall through a bell-shaped (non-linear) curve.

---
### Problem 4.3 &nbsp; ⭐⭐ &nbsp; Size classes with pd.cut

Create `size_class` from `farm_size_ha` with edges `0–1, 1–2, 2–5, >5` and labels `micro, small, medium, large`. Show the number of farms and the mean `yield_t_ha` per class. Then create quartile-based classes with `pd.qcut` and explain the difference between `cut` and `qcut`.

In [4]:
df = load_lesson_clean()
df["size_class"] = pd.cut(df["farm_size_ha"], bins=[0, 1, 2, 5, np.inf], labels=["micro", "small", "medium", "large"])
print(df.groupby("size_class", observed=True)["yield_t_ha"].agg(["count", "mean"]).round(2))

df["size_quartile"] = pd.qcut(df["farm_size_ha"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
print("\nqcut counts (equal sizes):", df["size_quartile"].value_counts().sort_index().tolist())
print("cut  counts (equal widths, unequal counts):", df["size_class"].value_counts().sort_index().tolist())

            count  mean
size_class             
micro        1081  2.33
small         646  2.45
medium        442  2.88
large         109  2.41

qcut counts (equal sizes): [570, 571, 568, 569]
cut  counts (equal widths, unequal counts): [1081, 646, 442, 109]


**💬 Why this works:** `cut` uses *fixed edges you choose* (meaningful thresholds → counts differ). `qcut` uses *quantiles* (equal counts → edges are learned from data, so in a pipeline use `KBinsDiscretizer` fitted on training data).

---
### Problem 4.4 &nbsp; ⭐⭐ &nbsp; Price features without cross-crop contamination

Using the daily price table (sorted by crop, date):

1. Compute the 1-day price change **wrongly** (`diff()` on the whole column) and **correctly** (`groupby('crop')`). Show where they differ.
2. For each crop compute `roll_mean_7`, `roll_std_30` and `lag_7`.
3. How many NaN does each new column have per crop at the start, and why?

In [5]:
p = prices.copy()
p["diff_wrong"] = p["price_ksh_kg"].diff()
p["diff_right"] = p.groupby("crop")["price_ksh_kg"].diff()
first_rows = p.groupby("crop").head(1)
print("First row of each crop - the wrong diff mixes crops, the right one is NaN:")
print(first_rows[["crop", "date", "diff_wrong", "diff_right"]].to_string(index=False))

g = p.groupby("crop")["price_ksh_kg"]
p["roll_mean_7"] = g.transform(lambda s: s.rolling(7).mean())
p["roll_std_30"] = g.transform(lambda s: s.rolling(30).std())
p["lag_7"]       = g.shift(7)
print("\nNaN per crop at the start:")
print(p.groupby("crop")[["roll_mean_7", "roll_std_30", "lag_7"]].apply(lambda d: d.isna().sum()))

First row of each crop - the wrong diff mixes crops, the right one is NaN:
  crop       date  diff_wrong  diff_right
 Beans 2019-01-01         NaN         NaN
 Maize 2019-01-01      -93.66         NaN
Potato 2019-01-01      -23.70         NaN

NaN per crop at the start:
        roll_mean_7  roll_std_30  lag_7
crop                                   
Beans           209          710     37
Maize           198          687     37
Potato          210          826     37


**💬 Why this works:** Without `groupby`, the first Beans price is subtracted from the last Potato price of the previous day: nonsense that raises no error. Rolling windows and lags *need history*, so the earliest rows are NaN. Notice the much bigger counts for `roll_std_30`: without `min_periods`, a rolling window is NaN if **any** price inside it is missing, so each missing day wipes out up to 30 outputs. That is why we clean/forward-fill the series *before* building rolling features.

---
### Problem 4.5 &nbsp; ⭐⭐⭐ &nbsp; Learn the seasonal calendar: from training years only

Fill and clean prices however you like (e.g. forward-fill per crop). Then, using **only 2019–2021**:

1. For each crop, compute a monthly *seasonal index* = mean price in that calendar month ÷ mean price in the same year (this removes the upward trend).
2. Report the peak ("lean season") month of each crop.
3. Repeat with **2022–2023** and check if the peak months agree. What does agreement tell you about using the month as a feature?

In [6]:
p = prices.copy()
p["price"] = p.groupby("crop")["price_ksh_kg"].ffill()
p["year"], p["month"] = p["date"].dt.year, p["date"].dt.month
p["ratio"] = p["price"] / p.groupby(["crop", "year"])["price"].transform("mean")

def peak_months(d):
    idx = d.groupby(["crop", "month"])["ratio"].mean().unstack()
    return idx.idxmax(axis=1), idx.max(axis=1).round(2)

train_peak, train_val = peak_months(p[p["year"] <= 2021])
later_peak, later_val = peak_months(p[p["year"] >= 2022])
print(pd.DataFrame({"peak month 2019-21": train_peak, "index": train_val,
                    "peak month 2022-23": later_peak, "index ": later_val}))

        peak month 2019-21  index  peak month 2022-23  index 
crop                                                         
Beans                    6   1.12                   5    1.16
Maize                    7   1.21                   8    1.22
Potato                   9   1.26                   9    1.24


**💬 Why this works:** The seasonal pattern learned from early years re-appears later (peaks agree to within a month), so *calendar month is a legitimate, stable predictor*. Note that the index was computed from training years only: computing it over the whole period would let the future shape your features (leakage).